<a href="https://colab.research.google.com/github/BenFreilichUVA/walmart-hurricane-project/blob/main/notebooks/team_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First, we need to clone the repo so we can actually pull the data and work on the project. Then, we need to load the raw data and put it into pandas DataFrames so the data can be manipulated and changed around. There are five raw files in the data folder that need to be converted.

'walmart_transactions.csv', turned into 'transactions'

'store_locations.csv', turned into 'stores'

'product_catalog.csv', turned into 'products'

'distribution_centers.csv', turned into 'centers'

'inventory_and_sales_db', turned into 'daily_sales' and 'inventory'




In [1]:
!rm -rf /content/walmart-hurricane-project
!git clone https://github.com/BenFreilichUVA/walmart-hurricane-project.git /content/walmart-hurricane-project

from pathlib import Path

def check_repo(root):
    root = Path(root)
    needed = ['CHARTER.md', 'README.md', 'data', 'notebooks/team_notebook.ipynb', 'notebooks/scratch']
    ok = True
    for item in needed:
        found = (root / item).exists()
        print('PASS' if found else 'FAIL', '-', item)
        ok = ok and found
    print()
    print('repo ready:', ok)
    return ok

MY_REPO = '/content/walmart-hurricane-project'
check_repo(MY_REPO)

Cloning into '/content/walmart-hurricane-project'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 43 (delta 13), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 962.16 KiB | 5.94 MiB/s, done.
Resolving deltas: 100% (13/13), done.
PASS - CHARTER.md
PASS - README.md
PASS - data
PASS - notebooks/team_notebook.ipynb
PASS - notebooks/scratch

repo ready: True


True

In [2]:
DATA = Path(MY_REPO) / 'data'

import pandas as pd
import sqlite3
transactions = pd.read_csv(DATA / 'walmart_transactions.csv')
stores       = pd.read_csv(DATA / 'store_locations.csv')
products     = pd.read_csv(DATA / 'product_catalog.csv')
centers          = pd.read_csv(DATA / 'distribution_centers.csv')

conn = sqlite3.connect(DATA / 'inventory_and_sales.db')

daily_sales = pd.read_sql('SELECT * FROM daily_sales_summary', conn)
inventory   = pd.read_sql('SELECT * FROM inventory_levels', conn)

conn.close()

print('transactions', transactions.shape)
print('stores      ', stores.shape)
print('products    ', products.shape)
print('centers     ', centers.shape)
print('daily_sales', daily_sales.shape)
print('inventory   ', inventory.shape)

decision_log = {}


transactions (25562, 8)
stores       (14, 8)
products     (45, 7)
centers          (5, 8)
daily_sales (3600, 5)
inventory    (4320, 6)


Next, cleaning has to be done. This will take a lot of different steps, and the assignment specifically asks to create a log of each change made.

In [3]:
clean = transactions.copy()

# Fixing the Store IDS for the transaction DataFrame
before = clean['store_id'].nunique()
clean['store_id'] = (clean['store_id']
                     .str.strip()
                     .str.upper()
                     .str.replace(r'^FL[-_]?', 'FL-', regex=True))
after = clean['store_id'].nunique()
# Assertions to ensure this step properly worked
assert clean['store_id'].nunique() == 12, 'expected 12 stores'
assert clean['store_id'].str.fullmatch(r'FL-\d{3}').all(), 'some IDs are not in FL-### format'
assert len(clean) == len(transactions), 'cleaning should not drop rows'
decision_log['transactions: store_id'] = f'Standardized store IDs to FL-###, removing unnecessary spacing and uppercasing ({before} spellings condensed to {after} spellings).'
# Fixing the Categories for the transaction DataFrame
before = clean['category'].nunique()
category_map = {
    'emergency': 'Emergency', 'emergency supplies': 'Emergency', 'grocery': 'Grocery', 'groceries': 'Grocery',
    'electronics': 'Electronics', 'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'household': 'Household', 'household essentials': 'Household', 'baby': 'Baby', 'baby products': 'Baby',
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast', 'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'snacks': 'Snacks', 'health': 'Health', 'health & wellness': 'Health',
}
clean['category'] = clean['category'].str.strip().str.lower().map(category_map)
after = clean['category'].nunique()
#Assertions to ensure this step properly worked
assert clean['category'].isna().sum() == 0, 'some categories did not map'
assert set(clean['category']) == set(daily_sales['category']), 'categories do not match the database'
decision_log['transactions: category'] = f'Mapped category names down to the 10 used in the database ({before} spellings condensed to {after} spellings).'

# Fixing the unit_price column to not be stored as text, and to instead be stored as a number
dollar_rows = clean['unit_price'].astype(str).str.contains('$', regex=False).sum()
before_type = clean['unit_price'].dtype

clean['unit_price'] = pd.to_numeric(
    clean['unit_price'].str.replace('$', '', regex=False).str.strip(),
    errors='coerce')

#Assertions to ensure this step properly worked
assert clean['unit_price'].isna().sum() == 0, 'some prices did not convert'
assert (clean['unit_price'] > 0).all(), 'found a zero or negative price'
decision_log['transactions: unit_price'] = f'Removed $ from {dollar_rows:,} prices and converted the column from text to numbers'

# Converting all of the timestamps into the same format

formats = ['%Y-%m-%d %H:%M:%S',   # 2024-09-15 18:36:16
           '%Y-%m-%dT%H:%M:%S',   # 2024-09-25T10:00:08
           '%m/%d/%Y %H:%M',      # 09/28/2024 17:19
           '%m-%d-%Y %H:%M:%S',   # 09-26-2024 17:46:57
           '%d-%b-%Y %H:%M']      # 13-Sep-2024 21:07

raw_ts = transactions['timestamp'].str.strip()
missing_ts = raw_ts.isna().sum()

parsed = pd.Series(pd.NaT, index=raw_ts.index, dtype='datetime64[ns]')
for f in formats:
    attempt = pd.to_datetime(raw_ts, format=f, errors='coerce')
    parsed = parsed.fillna(attempt)

clean['timestamp'] = parsed

unparsed = (clean['timestamp'].isna() & raw_ts.notna()).sum()
# Assertions to ensure the timestamps are correctly formatted.
assert unparsed == 0, 'some timestamps did not match any format'
assert clean['timestamp'].isna().sum() == missing_ts, 'blank count changed'
assert (clean['timestamp'].dropna().dt.month == 9).all(), 'found a date outside September'

decision_log['transactions: timestamp'] = (f'Parsed 5 timestamp formats into one datetime column '
                                           f'({missing_ts} rows had no timestamp, handled after duplicates were removed)')

# Deleting the duplicates

rows_before = len(clean)
dup_rows = clean.duplicated().sum()
dup_ids = clean['transaction_id'].duplicated().sum()

clean = clean.drop_duplicates()

# Assertions to ensure the duplicates were correctly dropped
assert len(clean) == rows_before - dup_rows, 'dropped the wrong number of rows'
assert clean['transaction_id'].is_unique, 'some transaction IDs still repeat'

decision_log['transactions: duplicates'] = (f'Dropped {dup_rows} exact duplicate rows '
                                            f'({rows_before:,} rows became {len(clean):,})')

# Fixing the quantities
rows_before = len(clean)
zero_rows = (clean['quantity'] == 0).sum()
neg_rows = (clean['quantity'] < 0).sum()
# Right now we are keeping the negative rows, assuming that they are returns.
clean = clean[clean['quantity'] != 0]
# Assertions to check that there are no zero quantities left in the dataframe.
assert (clean['quantity'] != 0).all(), 'zero quantities remain'
assert len(clean) == rows_before - zero_rows, 'dropped the wrong number of rows'

clean['is_return'] = clean['quantity'] < 0

decision_log['transactions: quantity'] = (f'Dropped {zero_rows} rows with quantity 0. '
                                          f'Kept {neg_rows} negative quantities as returns, flagged in is_return.')

# Creating a second table to only contain orders that have a timestamp

blank = clean['timestamp'].isna()
blank_rows = blank.sum()

clean_dated = clean[~blank].copy()
clean_dated['date'] = clean_dated['timestamp'].dt.normalize()
# Assertions to make sure the new table deleted all of the blank timestamps and that the dates are in the correct range.
assert clean_dated['timestamp'].isna().sum() == 0, 'blank timestamps remain'
assert len(clean_dated) == len(clean) - blank_rows, 'clean_dated has the wrong row count'
assert clean_dated['date'].between('2024-09-01', '2024-09-30').all(), 'date outside September'

decision_log['transactions: blank timestamps'] = (
    f'Kept {blank_rows} rows with no timestamp in clean ({blank_rows / len(clean):.1%} of rows). '
    f'Left them out of clean_dated, which is used for daily counts (Q1, Q3). ')

for step, note in decision_log.items():
    print(f'{step} - {note}')


transactions: store_id - Standardized store IDs to FL-###, removing unnecessary spacing and uppercasing (60 spellings condensed to 12 spellings).
transactions: category - Mapped category names down to the 10 used in the database (37 spellings condensed to 10 spellings).
transactions: unit_price - Removed $ from 3,029 prices and converted the column from text to numbers
transactions: timestamp - Parsed 5 timestamp formats into one datetime column (886 rows had no timestamp, handled after duplicates were removed)
transactions: duplicates - Dropped 30 exact duplicate rows (25,562 rows became 25,532)
transactions: quantity - Dropped 0 rows with quantity 0. Kept 130 negative quantities as returns, flagged in is_return.
transactions: blank timestamps - Kept 884 rows with no timestamp in clean (3.5% of rows). Left them out of clean_dated, which is used for daily counts (Q1, Q3). 


Looking through the stores file, we see that there are problems including messy IDS, duplicate rows, and overall completeness errors. These will be fixed in the cell below.